In [ ]:
"""
================================================================================
  MLP Training — FER-2013  (7 classes, all emotions)

  Baseline = faithful reproduction of the confirmed 48.38% script:
    - Custom FER2013Dataset (not ImageFolder)
    - 90/10 train/val split from train folder; test folder = held-out test
    - Architecture: 1024 → 512 → 256 → 7  (3M params)
    - Dropout: 0.25 / 0.25 / 0.2
    - AdamW  lr=0.001  wd=5e-5
    - LabelSmoothingCrossEntropy  smoothing=0.1
    - Warmup 5 ep → cosine 75 ep  (100 ep total)
    - Patience 20, min_delta 0.05
    - NO MixUp, NO WeightedRandomSampler

  Variants: one axis changed vs baseline at a time.
  Each variant × each seed = one full training run.
  Every epoch printed: train loss/acc | val loss/acc | test loss/acc
================================================================================
"""

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])

import os, json, time, random
from pathlib import Path
from copy import deepcopy
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from PIL import Image

# ══════════════════════════════════════════════════════════════════════════════
#  SEEDS
# ══════════════════════════════════════════════════════════════════════════════
SEEDS = [42, 123, 7]

# ══════════════════════════════════════════════════════════════════════════════
#  VARIANTS — baseline first, then one axis changed at a time
# ══════════════════════════════════════════════════════════════════════════════
VARIANTS = [
    # ── BASELINE — exact original 48.38% config ───────────────────────────
    {
        "name":         "baseline",
        "hidden":       [1024, 512, 256],
        "dropout":      [0.25, 0.25, 0.20],
        "lr":           1e-3,
        "weight_decay": 5e-5,
        "smoothing":    0.1,
        "batch_size":   128,
        "max_weight":   2.5,       # class weight cap
        "warmup_ep":    5,
        "max_ep":       100,
        "patience":     20,
        "min_delta":    0.05,
    },
    # ── Wider + deeper architecture ────────────────────────────────────────
    {
        "name":         "wider_deeper",
        "hidden":       [2048, 1024, 512, 256],
        "dropout":      [0.35, 0.30, 0.25, 0.20],
        "lr":           1e-3,
        "weight_decay": 5e-5,
        "smoothing":    0.1,
        "batch_size":   128,
        "max_weight":   2.5,
        "warmup_ep":    5,
        "max_ep":       120,
        "patience":     20,
        "min_delta":    0.05,
    },
    # ── Lower LR ───────────────────────────────────────────────────────────
    {
        "name":         "lower_lr",
        "hidden":       [1024, 512, 256],
        "dropout":      [0.25, 0.25, 0.20],
        "lr":           5e-4,
        "weight_decay": 5e-5,
        "smoothing":    0.1,
        "batch_size":   128,
        "max_weight":   2.5,
        "warmup_ep":    5,
        "max_ep":       120,
        "patience":     20,
        "min_delta":    0.05,
    },
    # ── Higher dropout ─────────────────────────────────────────────────────
    {
        "name":         "high_dropout",
        "hidden":       [1024, 512, 256],
        "dropout":      [0.40, 0.35, 0.30],
        "lr":           1e-3,
        "weight_decay": 5e-5,
        "smoothing":    0.1,
        "batch_size":   128,
        "max_weight":   2.5,
        "warmup_ep":    5,
        "max_ep":       120,
        "patience":     20,
        "min_delta":    0.05,
    },
    # ── Higher weight decay (L2) ───────────────────────────────────────────
    {
        "name":         "high_wd",
        "hidden":       [1024, 512, 256],
        "dropout":      [0.25, 0.25, 0.20],
        "lr":           1e-3,
        "weight_decay": 1e-3,
        "smoothing":    0.1,
        "batch_size":   128,
        "max_weight":   2.5,
        "warmup_ep":    5,
        "max_ep":       120,
        "patience":     20,
        "min_delta":    0.05,
    },
]

# ══════════════════════════════════════════════════════════════════════════════
#  FIXED SETTINGS
# ══════════════════════════════════════════════════════════════════════════════
CLASSES     = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
NUM_CLASSES = 7
DATA_DIR    = "/content"   # expects /content/train and /content/test
OUT_DIR     = "/content"
NUM_WORKERS = 2
GRAD_CLIP   = 1.0
VAL_SPLIT   = 0.1          # 90/10 from train folder — matches original

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{'='*65}")
print(f"  MLP — FER-2013 (7-class)  |  Device: {DEVICE}")
print(f"  Seeds   : {SEEDS}")
print(f"  Variants: {len(VARIANTS)}  (baseline first)")
print(f"  Total runs: {len(VARIANTS) * len(SEEDS)}")
print(f"{'='*65}")


# ══════════════════════════════════════════════════════════════════════════════
#  DOWNLOAD  (matches original script)
# ══════════════════════════════════════════════════════════════════════════════
def download_dataset():
    train_dir = os.path.join(DATA_DIR, "train")
    test_dir  = os.path.join(DATA_DIR, "test")
    if os.path.isdir(train_dir) and os.path.isdir(test_dir):
        print(f"✅ Dataset found — {sorted(os.listdir(train_dir))}"); return
    print("⚠️  Dataset not found. Downloading...")
    kdir  = os.path.expanduser("~/.kaggle")
    kfile = os.path.join(kdir, "kaggle.json")
    os.makedirs(kdir, exist_ok=True)
    if not os.path.exists(kfile):
        # Option A: from Drive
        # from google.colab import drive; drive.mount('/content/drive')
        # import shutil; shutil.copy('/content/drive/MyDrive/kaggle.json', kfile)
        # Option B: inline
        creds = {"username": "patilatharv", "key": "YOUR_API_KEY_HERE"}
        with open(kfile, "w") as f: json.dump(creds, f)
    os.chmod(kfile, 0o600)
    subprocess.run(["kaggle","datasets","download","-d","msambare/fer2013",
                    "--unzip","-p",DATA_DIR], check=True)
    print("✅ Done")

download_dataset()


# ══════════════════════════════════════════════════════════════════════════════
#  DATASET — identical to original script
# ══════════════════════════════════════════════════════════════════════════════
class FER2013Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir  = Path(root_dir)
        self.transform = transform
        self.classes   = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples   = []
        for cls in self.classes:
            for img_path in (self.root_dir / cls).glob("*.jpg"):
                self.samples.append((img_path, self.class_to_idx[cls]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("L")
        if self.transform: img = self.transform(img)
        return img, label

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(12),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

full_train_ds = FER2013Dataset(os.path.join(DATA_DIR, "train"), transform=transform_train)
test_ds       = FER2013Dataset(os.path.join(DATA_DIR, "test"),  transform=transform_test)

# Fixed 90/10 val split from train folder — same seed across all runs
train_size = int((1 - VAL_SPLIT) * len(full_train_ds))
val_size   = len(full_train_ds) - train_size
train_ds, val_ds = random_split(
    full_train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"\n📊 Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}")

def compute_class_weights(dataset, max_weight=2.5):
    """Identical to original compute_balanced_weights."""
    if hasattr(dataset, "samples"):
        labels = [l for _, l in dataset.samples]
    else:
        labels = [dataset.dataset.samples[i][1] for i in dataset.indices]
    counter = Counter(labels)
    total   = len(labels)
    weights = torch.zeros(NUM_CLASSES)
    for ci, cnt in counter.items():
        weights[ci] = min(total / (len(counter) * cnt), max_weight)
    return weights


# ══════════════════════════════════════════════════════════════════════════════
#  MODEL
# ══════════════════════════════════════════════════════════════════════════════
class MLP(nn.Module):
    def __init__(self, input_size, hidden_dims, num_classes, dropouts):
        super().__init__()
        assert len(dropouts) == len(hidden_dims)
        layers = [nn.Flatten()]
        in_d   = input_size
        for h, d in zip(hidden_dims, dropouts):
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(d)]
            in_d = h
        layers.append(nn.Linear(in_d, num_classes))
        self.net = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x): return self.net(x)

    @property
    def n_params(self): return sum(p.numel() for p in self.parameters())


# ══════════════════════════════════════════════════════════════════════════════
#  LOSS — identical to original LabelSmoothingCrossEntropy
# ══════════════════════════════════════════════════════════════════════════════
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.weight    = weight

    def forward(self, pred, target):
        n  = pred.size(1)
        oh = torch.zeros_like(pred).scatter(1, target.unsqueeze(1), 1)
        oh = oh * (1 - self.smoothing) + self.smoothing / n
        lp = torch.nn.functional.log_softmax(pred, dim=1)
        if self.weight is not None:
            loss = -(oh * lp * self.weight.unsqueeze(0)).sum(dim=1)
        else:
            loss = -(oh * lp).sum(dim=1)
        return loss.mean()


# ══════════════════════════════════════════════════════════════════════════════
#  TRAIN / EVAL
# ══════════════════════════════════════════════════════════════════════════════
def train_epoch(model, loader, crit, opt):
    model.train()
    tl, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        out  = model(x)
        loss = crit(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        tl      += loss.item() * x.size(0)
        correct += out.max(1)[1].eq(y).sum().item()
        total   += y.size(0)
    return tl / total, 100.0 * correct / total

@torch.no_grad()
def eval_epoch(model, loader, crit):
    model.eval()
    tl, correct, total = 0.0, 0, 0
    pa, la = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        tl     += crit(out, y).item() * x.size(0)
        p       = out.max(1)[1]
        correct += p.eq(y).sum().item()
        total   += y.size(0)
        pa.extend(p.cpu().numpy()); la.extend(y.cpu().numpy())
    acc = 100.0 * correct / total
    ap, al = np.array(pa), np.array(la)
    pc = {c: 100.0 * float((ap[al==i]==i).sum()) / max(int((al==i).sum()), 1)
          for i, c in enumerate(CLASSES)}
    return tl / total, acc, pc, pa, la


# ══════════════════════════════════════════════════════════════════════════════
#  SINGLE RUN — one (variant, seed) pair, prints every epoch
# ══════════════════════════════════════════════════════════════════════════════
def run_single(cfg, seed, run_label):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    # DataLoaders — plain shuffle, no WeightedSampler (matches original)
    tr_ldr  = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                         drop_last=True)
    val_ldr = DataLoader(val_ds,   batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    tst_ldr = DataLoader(test_ds,  batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

    model = MLP(48*48, cfg["hidden"], NUM_CLASSES, cfg["dropout"]).to(DEVICE)

    cw   = compute_class_weights(train_ds, max_weight=cfg["max_weight"]).to(DEVICE)
    crit = LabelSmoothingCE(smoothing=cfg["smoothing"], weight=cw)
    opt  = optim.AdamW(model.parameters(), lr=cfg["lr"],
                       weight_decay=cfg["weight_decay"], betas=(0.9, 0.999))

    # LR schedule: warmup then cosine — identical to original lr_lambda
    wu = cfg["warmup_ep"]
    me = cfg["max_ep"]
    def lr_lambda(ep):
        if ep < wu:   return (ep + 1) / wu
        return 0.5 * (1 + np.cos(np.pi * (ep - wu) / max(me - wu, 1)))
    sched = optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    ckpt    = os.path.join(OUT_DIR, f"mlp_{cfg['name']}_seed{seed}_best.pt")
    history = {k: [] for k in ["train_loss","train_acc","val_loss","val_acc",
                                "test_loss","test_acc","lr"]}
    best_val, best_ep, patience_ctr = 0.0, 0, 0
    best_state = None

    print(f"\n  {'─'*84}")
    print(f"  {run_label}")
    print(f"  params={model.n_params:,}  "
          f"class_weights={[f'{v:.2f}' for v in cw.cpu().numpy()]}")
    print(f"  {'─'*84}")
    print(f"  {'Ep':>5}  {'LR':>9}  "
          f"{'Tr Loss':>9} {'Tr Acc':>8}  "
          f"{'Va Loss':>9} {'Va Acc':>8}  "
          f"{'Te Loss':>9} {'Te Acc':>8}")
    print(f"  {'─'*84}")

    t0 = time.time()
    for epoch in range(cfg["max_ep"]):
        lr_now = opt.param_groups[0]["lr"]
        trl, tra             = train_epoch(model, tr_ldr, crit, opt)
        vll, vla, vpc, _, _  = eval_epoch(model, val_ldr, crit)
        tel, tea, tpc, _, _  = eval_epoch(model, tst_ldr, crit)
        sched.step()

        history["train_loss"].append(trl); history["train_acc"].append(tra)
        history["val_loss"].append(vll);   history["val_acc"].append(vla)
        history["test_loss"].append(tel);  history["test_acc"].append(tea)
        history["lr"].append(lr_now)

        mark = ""
        # min_delta matches original: only update best if improvement > threshold
        if vla > best_val + cfg["min_delta"]:
            best_val  = vla
            best_ep   = epoch
            best_state = deepcopy(model.state_dict())
            torch.save(best_state, ckpt)
            mark = " ★"
            patience_ctr = 0
        else:
            patience_ctr += 1

        # Every epoch printed
        print(f"  {epoch+1:>5}  {lr_now:>9.2e}  "
              f"{trl:>9.4f} {tra:>7.2f}%  "
              f"{vll:>9.4f} {vla:>7.2f}%  "
              f"{tel:>9.4f} {tea:>7.2f}%{mark}")

        if patience_ctr >= cfg["patience"]:
            print(f"\n  ⏹  Early stop @ ep {epoch+1}  "
                  f"(best val {best_val:.2f}% @ ep {best_ep+1})")
            break

    print(f"\n  ✅ Done in {(time.time()-t0)/60:.1f} min")

    # Reload best and get final metrics
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    _, val_acc,  val_pc,  _,         _          = eval_epoch(model, val_ldr,  crit)
    _, test_acc, test_pc, all_preds, all_labels  = eval_epoch(model, tst_ldr, crit)

    print(f"\n  📊 Best checkpoint (ep {best_ep+1}):")
    print(f"     Val  {val_acc:.2f}%    Test  {test_acc:.2f}%")
    print(f"\n  Per-class test accuracy:")
    for cls in CLASSES:
        s = "✅" if test_pc[cls]>40 else "⚠️ " if test_pc[cls]>20 else "❌"
        print(f"    {cls:<10}: val={val_pc[cls]:5.1f}%  test={test_pc[cls]:5.1f}%  {s}")
    print(f"\n{classification_report(all_labels, all_preds, target_names=CLASSES, digits=3)}")

    result = {
        "model": "MLP", "variant": cfg["name"], "seed": seed,
        "dataset": "fer7",
        "classes": CLASSES, "num_classes": NUM_CLASSES,
        "test_acc": float(test_acc), "val_acc": float(val_acc),
        "best_val_acc": float(best_val), "best_epoch": int(best_ep + 1),
        "n_params": int(model.n_params),
        "per_class_acc":     {k: float(v) for k, v in test_pc.items()},
        "per_class_acc_val": {k: float(v) for k, v in val_pc.items()},
        "history":   {k: [float(x) for x in v] for k, v in history.items()},
        "all_preds": [int(x) for x in all_preds],
        "all_labels":[int(x) for x in all_labels],
        "config":    {k: (list(v) if isinstance(v,(list,tuple)) else v)
                      for k, v in cfg.items()},
    }
    with open(os.path.join(OUT_DIR, f"mlp_{cfg['name']}_seed{seed}_results.json"), "w") as f:
        json.dump(result, f, indent=2)
    return result, history


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN LOOP — variants × seeds, baseline always runs first
# ══════════════════════════════════════════════════════════════════════════════
all_runs   = {cfg["name"]: [] for cfg in VARIANTS}
total_runs = len(VARIANTS) * len(SEEDS)
run_idx    = 0

for cfg in VARIANTS:
    print(f"\n{'═'*90}")
    print(f"  VARIANT: {cfg['name'].upper()}")
    print(f"  hidden={cfg['hidden']}  dropout={cfg['dropout']}")
    print(f"  lr={cfg['lr']}  wd={cfg['weight_decay']}  bs={cfg['batch_size']}  "
          f"smoothing={cfg['smoothing']}  max_ep={cfg['max_ep']}")
    print(f"  Seeds: {SEEDS}")
    print(f"{'═'*90}")

    for seed in SEEDS:
        run_idx += 1
        label = (f"Run {run_idx}/{total_runs} — "
                 f"variant={cfg['name']}  seed={seed}")
        result, history = run_single(cfg, seed, label)
        all_runs[cfg["name"]].append((result, history))


# ══════════════════════════════════════════════════════════════════════════════
#  AGGREGATE — mean ± std per variant across seeds
# ══════════════════════════════════════════════════════════════════════════════
summary = {}
for vname, runs in all_runs.items():
    if not runs: continue
    test_accs = [r["test_acc"] for r, _ in runs]
    val_accs  = [r["val_acc"]  for r, _ in runs]
    best_eps  = [r["best_epoch"] for r, _ in runs]
    pc_means  = {c: float(np.mean([r["per_class_acc"][c] for r, _ in runs]))
                 for c in CLASSES}
    summary[vname] = {
        "test_mean":      float(np.mean(test_accs)),
        "test_std":       float(np.std(test_accs)),
        "val_mean":       float(np.mean(val_accs)),
        "val_std":        float(np.std(val_accs)),
        "ep_mean":        float(np.mean(best_eps)),
        "per_class_mean": pc_means,
        "n_params":       runs[0][0]["n_params"],
        "all_test_accs":  test_accs,
    }

best_run = max(
    [r for runs in all_runs.values() for r, _ in runs],
    key=lambda r: r["test_acc"]
)
print(f"\n  🏆 Best run: variant={best_run['variant']}  "
      f"seed={best_run['seed']}  test={best_run['test_acc']:.2f}%")

with open(os.path.join(OUT_DIR, "mlp_fer7_results.json"), "w") as f:
    json.dump(best_run, f, indent=2)
print(f"  💾 Best → /content/mlp_fer7_results.json  (use with compare.py)")


# ══════════════════════════════════════════════════════════════════════════════
#  FINAL TABLES
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*75}")
print(f"  MLP VARIANTS — MEAN ± STD  (seeds={SEEDS})")
print(f"{'═'*75}")
print(f"  {'Variant':<18}  {'Val':>15}  {'Test':>15}  {'~Ep':>6}  {'Params':>10}")
print(f"  {'─'*65}")
for vname, s in summary.items():
    mark = " ←" if vname == best_run["variant"] else ""
    print(f"  {vname:<18}  "
          f"{s['val_mean']:>6.2f}±{s['val_std']:>4.2f}%  "
          f"{s['test_mean']:>6.2f}±{s['test_std']:>4.2f}%  "
          f"{s['ep_mean']:>6.0f}  {s['n_params']:>10,}{mark}")
print(f"\n  Random baseline: {100/NUM_CLASSES:.1f}%")

print(f"\n  Per-seed test accuracies:")
print(f"  {'Variant':<18}" + "".join(f"  seed={sd}" for sd in SEEDS))
print(f"  {'─'*55}")
for vname, runs in all_runs.items():
    if not runs: continue
    row = f"  {vname:<18}"
    for r, _ in runs: row += f"  {r['test_acc']:>8.2f}%"
    print(row)

print(f"\n  Per-class test accuracy (mean):")
print(f"  {'Class':<12}" + "".join(f"  {n:>16}" for n in summary))
print(f"  {'─'*80}")
for cls in CLASSES:
    row = f"  {cls:<12}"
    for s in summary.values():
        row += f"  {s['per_class_mean'][cls]:>15.1f}%"
    print(row)


# ══════════════════════════════════════════════════════════════════════════════
#  PLOTS
# ══════════════════════════════════════════════════════════════════════════════
VCOLS      = plt.cm.tab10(np.linspace(0, 0.6, len(VARIANTS)))
SEED_LINES = ["-", "--", ":"]

# ── 1. All val+test curves overlaid ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("MLP — Val & Test Accuracy (all variants × all seeds)", fontsize=13, fontweight="bold")
for vi, (cfg, color) in enumerate(zip(VARIANTS, VCOLS)):
    for si, (seed, ls) in enumerate(zip(SEEDS, SEED_LINES)):
        r, h = all_runs[cfg["name"]][si]
        epx  = list(range(1, len(h["val_acc"]) + 1))
        axes[0].plot(epx, h["val_acc"],  color=color, ls=ls, lw=1.5, alpha=0.85,
                     label=cfg["name"] if si==0 else None)
        axes[1].plot(epx, h["test_acc"], color=color, ls=ls, lw=1.5, alpha=0.85,
                     label=cfg["name"] if si==0 else None)
for ax, title in zip(axes, ["Validation Accuracy", "Test Accuracy"]):
    ax.axhline(100/NUM_CLASSES, color="gray", ls=":", alpha=0.5,
               label=f"Random {100/NUM_CLASSES:.0f}%")
    ax.set(title=title, xlabel="Epoch", ylabel="%")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
for si, (seed, ls) in enumerate(zip(SEEDS, SEED_LINES)):
    axes[0].plot([], [], color="gray", ls=ls, label=f"seed={seed}")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.savefig("/content/mlp_all_curves.png", dpi=150, bbox_inches="tight")
plt.show(); print("📸 /content/mlp_all_curves.png")


# ── 2. Per-variant loss + acc (seeds overlaid) ────────────────────────────
n_v = len(VARIANTS)
fig, axes = plt.subplots(n_v, 2, figsize=(16, 4 * n_v))
if n_v == 1: axes = [axes]
fig.suptitle("MLP — Loss & Accuracy per Variant (seeds overlaid)", fontsize=12, fontweight="bold")
for vi, (cfg, color) in enumerate(zip(VARIANTS, VCOLS)):
    vname = cfg["name"]
    for si, (seed, ls) in enumerate(zip(SEEDS, SEED_LINES)):
        r, h = all_runs[vname][si]
        epx  = list(range(1, len(h["train_loss"]) + 1))
        axes[vi][0].plot(epx, h["train_loss"], color="#2563eb", ls=ls, lw=1.2, alpha=0.7)
        axes[vi][0].plot(epx, h["val_loss"],   color="#f59e0b", ls=ls, lw=1.2, alpha=0.7)
        axes[vi][0].plot(epx, h["test_loss"],  color="#dc2626", ls=ls, lw=1.2, alpha=0.7)
        axes[vi][1].plot(epx, h["train_acc"],  color="#2563eb", ls=ls, lw=1.2, alpha=0.7,
                         label="Train" if si==0 else None)
        axes[vi][1].plot(epx, h["val_acc"],    color="#f59e0b", ls=ls, lw=1.2, alpha=0.7,
                         label="Val"   if si==0 else None)
        axes[vi][1].plot(epx, h["test_acc"],   color="#dc2626", ls=ls, lw=1.2, alpha=0.7,
                         label="Test"  if si==0 else None)
    s = summary[vname]
    axes[vi][0].set(title=f"{vname} — Loss", xlabel="Epoch", ylabel="Loss")
    axes[vi][0].grid(alpha=0.3)
    axes[vi][1].axhline(100/NUM_CLASSES, color="gray", ls=":", alpha=0.4)
    axes[vi][1].set(title=f"{vname} — Acc  "
                          f"test {s['test_mean']:.2f}±{s['test_std']:.2f}%",
                    xlabel="Epoch", ylabel="%")
    axes[vi][1].legend(fontsize=8); axes[vi][1].grid(alpha=0.3)
axes[0][0].plot([], [], color="#2563eb", label="Train")
axes[0][0].plot([], [], color="#f59e0b", label="Val")
axes[0][0].plot([], [], color="#dc2626", label="Test")
axes[0][0].legend(fontsize=8)
plt.tight_layout()
plt.savefig("/content/mlp_variant_curves.png", dpi=150, bbox_inches="tight")
plt.show(); print("📸 /content/mlp_variant_curves.png")


# ── 3. Mean ± std bar + per-class of best ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("MLP — Mean ± Std Test Accuracy + Per-class of Best", fontsize=12, fontweight="bold")
vnames = list(summary.keys())
means  = [summary[v]["test_mean"] for v in vnames]
stds   = [summary[v]["test_std"]  for v in vnames]
bcolors = ["#16a34a" if v == best_run["variant"] else "#2563eb" for v in vnames]
bars = axes[0].bar(vnames, means, yerr=stds, color=bcolors, edgecolor="white",
                   capsize=5, error_kw={"elinewidth":2})
for bar, m, s in zip(bars, means, stds):
    axes[0].text(bar.get_x()+bar.get_width()/2, m+s+0.3,
                 f"{m:.1f}±{s:.1f}%", ha="center", fontsize=8, fontweight="bold")
axes[0].axhline(100/NUM_CLASSES, color="gray", ls="--", alpha=0.6,
                label=f"Random {100/NUM_CLASSES:.0f}%")
axes[0].set(title="Test Accuracy (mean ± std)", ylabel="%",
            ylim=(0, max(means)+max(stds)+12))
axes[0].set_xticklabels(vnames, rotation=20, ha="right")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)
best_pc  = summary[best_run["variant"]]["per_class_mean"]
cls_accs = [best_pc[c] for c in CLASSES]
bcolors2 = ["#16a34a" if a>40 else "#ca8a04" if a>20 else "#dc2626" for a in cls_accs]
bars2 = axes[1].bar(CLASSES, cls_accs, color=bcolors2, edgecolor="white")
for bar, a in zip(bars2, cls_accs):
    axes[1].text(bar.get_x()+bar.get_width()/2, a+0.5, f"{a:.1f}%",
                 ha="center", fontsize=8, fontweight="bold")
axes[1].axhline(100/NUM_CLASSES, color="gray", ls="--", alpha=0.6, label="Random")
axes[1].set(title=f"Per-class — {best_run['variant']} (best, mean)",
            ylabel="%", ylim=(0, 110))
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/content/mlp_summary_bars.png", dpi=150, bbox_inches="tight")
plt.show(); print("📸 /content/mlp_summary_bars.png")


# ── 4. Confusion matrix — best run ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
cm = confusion_matrix(best_run["all_labels"], best_run["all_preds"], normalize="true")
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
    ax=ax, cmap="Blues", values_format=".2f")
ax.set_title(f"MLP Best — variant={best_run['variant']}  seed={best_run['seed']}\n"
             f"test={best_run['test_acc']:.2f}%  ep={best_run['best_epoch']}",
             fontsize=10, fontweight="bold")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig("/content/mlp_best_confusion.png", dpi=150, bbox_inches="tight")
plt.show(); print("📸 /content/mlp_best_confusion.png")

print(f"\n✅ All MLP outputs in /content/")
print(f"   mlp_all_curves.png       ← all variants × seeds")
print(f"   mlp_variant_curves.png   ← per variant, seeds overlaid")
print(f"   mlp_summary_bars.png     ← mean±std + per-class")
print(f"   mlp_best_confusion.png   ← confusion matrix")
print(f"   mlp_fer7_results.json    ← best run → use with compare.py")

  MLP — FER-2013 (7-class)  |  Device: cuda
  Seeds   : [42, 123, 7]
  Variants: 5  (baseline first)
  Total runs: 15
⚠️  Dataset not found. Downloading...
✅ Done

📊 Train: 25,838  |  Val: 2,871  |  Test: 7,178

══════════════════════════════════════════════════════════════════════════════════════════
  VARIANT: BASELINE
  hidden=[1024, 512, 256]  dropout=[0.25, 0.25, 0.2]
  lr=0.001  wd=5e-05  bs=128  smoothing=0.1  max_ep=100
  Seeds: [42, 123, 7]
══════════════════════════════════════════════════════════════════════════════════════════

  ────────────────────────────────────────────────────────────────────────────────────
  Run 1/15 — variant=baseline  seed=42
  params=3,021,831  class_weights=['1.02', '2.50', '1.00', '0.57', '0.83', '0.85', '1.29']
  ────────────────────────────────────────────────────────────────────────────────────
     Ep         LR    Tr Loss   Tr Acc    Va Loss   Va Acc    Te Loss   Te Acc
  ─────────────────────────────────────────────────────────────────────